In [ ]:
import os
import numpy as np
import pandas as pd
import pymannkendall as mkpkg
from scipy.stats import spearmanr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

In [ ]:

# Paths
NATIONAL_PATH = "./data/national_data.csv"
OUTPATH = "./output_figures_python"
os.makedirs(OUTPATH, exist_ok=True)

YEARS = [2020, 2021, 2022, 2023]


# Style
plt.rcParams["font.family"] = "serif"
plt.rcParams["axes.linewidth"] = 0.9

YR_COL = {2020: "#d73027", 2021: "#f4a442", 2022: "#4dac26", 2023: "#4575b4"}
KW_COL = {"Measles": "#2166ac", "Tigdas": "#1a9641", "Tipdas": "#d7191c"}
KW_LABEL = {"Measles": "Measles (EN)", "Tigdas": "Tigdas (FIL)", "Tipdas": "Tipdas (FIL)"}


def sig_stars(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""


# 1. LOAD & CLEAN NATIONAL DATA

def load_national(path):
    """Parses the Week + 4-years-of-4-columns national CSV, computes weekly
    incidence as the first difference of cumulative cases (clipped at 0)."""
    raw = pd.read_csv(path, header=None, skiprows=2)

    frames = []
    for i, yr in enumerate(YEARS):
        ci = i * 4 + 1  # 0-indexed: col 0 = Week
        df = pd.DataFrame({
            "Year": yr,
            "Week": raw[0].astype(int),
            "Cases": pd.to_numeric(raw[ci], errors="coerce"),
            "Measles": pd.to_numeric(raw[ci + 1], errors="coerce"),
            "Tipdas": pd.to_numeric(raw[ci + 2], errors="coerce"),
            "Tigdas": pd.to_numeric(raw[ci + 3], errors="coerce"),
        })
        frames.append(df)
    nat = pd.concat(frames, ignore_index=True)

    nat["inc"] = np.nan
    for yr in YEARS:
        m = nat["Year"] == yr
        c = nat.loc[m, "Cases"].values
        d = np.diff(c, prepend=np.nan)
        d = np.where(d < 0, 0, d)
        nat.loc[m, "inc"] = d
    nat = nat.dropna(subset=["inc"]).reset_index(drop=True)
    return nat


nat = load_national(NATIONAL_PATH)
print("Rows per year after cleaning:")
print(nat.groupby("Year").size())

# 2. MANN-KENDALL TREND ANALYSIS (national, per year)

mk_rows = []
for yr in YEARS:
    x = nat.loc[nat.Year == yr, "inc"].values
    res = mkpkg.original_test(x)
    trend = ("Increasing" if (res.Tau > 0 and res.p < 0.05)
             else "Decreasing" if (res.Tau < 0 and res.p < 0.05)
             else "No trend")
    mk_rows.append({"Year": yr, "tau": res.Tau, "p_value": res.p,
                     "slope": res.slope, "trend": trend})
mk_df = pd.DataFrame(mk_rows)
print("\nMann-Kendall results:")
print(mk_df.to_string(index=False))

# 3. SPEARMAN CROSS-CORRELATION (national, lags -1/0/+1)

def sp_lag(search, cases, lag):
    """Spearman correlation at a given lag. lag>0: search leads (shifted
    forward relative to cases); lag<0: search lags cases."""
    n = len(search)
    if lag > 0:
        s, c = search[:n - lag], cases[lag:]
    elif lag < 0:
        s, c = search[abs(lag):], cases[:n - abs(lag)]
    else:
        s, c = search, cases
    valid = (~np.isnan(s)) & (~np.isnan(c)) & (np.nanstd(s) > 0)
    if valid.sum() < 5:
        return np.nan, np.nan
    rho, p = spearmanr(s[valid], c[valid])
    return rho, p


sp_rows = []
for yr in YEARS:
    sub = nat[nat.Year == yr]
    inc = sub["inc"].values
    for kw in ["Measles", "Tigdas", "Tipdas"]:
        rsv = sub[kw].values
        for lag in [-1, 0, 1]:
            rho, p = sp_lag(rsv, inc, lag)
            sp_rows.append({"Year": yr, "Keyword": kw, "Lag": lag, "rho": rho, "p": p})
sp_df = pd.DataFrame(sp_rows)
sp_df["sig"] = sp_df["p"].apply(sig_stars)
print("\nSpearman cross-correlation (all lags):")
print(sp_df.to_string(index=False))

# best (max |rho|) lag per keyword-year -- used for Figure 2
best = sp_df.loc[sp_df.groupby(["Year", "Keyword"])["rho"].apply(lambda s: s.abs().idxmax())].reset_index(drop=True)
print("\nBest lag per keyword-year:")
print(best.to_string(index=False))

# FIGURE 1 -- Mann-Kendall bar + heatmap

print("\nGenerating Figure 1...")

fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
x = np.arange(len(YEARS))

ax = axes[0]
bar_colors = [{"Increasing": "#1a9641", "Decreasing": "#d7191c"}.get(r.trend, "#b0b0b0")
              for _, r in mk_df.iterrows()]
ax.bar(x, mk_df["tau"], width=0.55, color=bar_colors, edgecolor="black", linewidth=0.9)
ax.axhline(0, color="black", linewidth=1.1)
for i, r in mk_df.iterrows():
    lbl = f"{r.trend[:3].upper()} {sig_stars(r.p_value)}".strip()
    va = "bottom" if r.tau >= 0 else "top"
    off = 0.03 if r.tau >= 0 else -0.03
    ax.text(i, r.tau + off, lbl, ha="center", va=va, fontsize=9, fontweight="bold")
    if abs(r.tau) > 0.12:
        ax.text(i, r.tau / 2, f"{r.slope:+.2f}\ncases/wk", ha="center", va="center",
                fontsize=8, color="white", fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(YEARS)
ax.set_ylim(-0.55, 0.45)
ax.set_xlabel("Year"); ax.set_ylabel("Kendall's \u03c4")
ax.set_title("(a)  Kendall's \u03c4 by Year", fontsize=12, fontweight="bold")
ax.grid(axis="y", linestyle="--", color="#cccccc", linewidth=0.5)
ax.set_axisbelow(True)
handles = [Rectangle((0, 0), 1, 1, color=c) for c in ["#1a9641", "#d7191c", "#b0b0b0"]]
ax.legend(handles, ["Increasing \u2014 p<0.05", "Decreasing \u2014 p<0.05", "Not significant"],
          loc="lower center", bbox_to_anchor=(0.5, -0.34), fontsize=8, frameon=True)

ax2 = axes[1]
taus = mk_df["tau"].values.reshape(1, -1)
im = ax2.imshow(taus, cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")
ax2.set_xticks(x); ax2.set_xticklabels(YEARS)
ax2.set_yticks([0]); ax2.set_yticklabels(["National\nLevel"], fontweight="bold")
for i, r in mk_df.iterrows():
    txt = f"\u03c4 = {r.tau:+.2f}{' (sig)' if r.p_value < 0.05 else ''}\n{r.trend}"
    color = "white" if abs(r.tau) > 0.33 else "black"
    ax2.text(i, 0, txt, ha="center", va="center", fontsize=9, fontweight="bold", color=color)
ax2.set_title("(b)  Summary Heatmap", fontsize=12, fontweight="bold")
plt.colorbar(im, ax=ax2, fraction=0.05, pad=0.04).set_label("Kendall's \u03c4")

fig.suptitle("Figure 1. Mann-Kendall Trend Analysis of National Weekly Measles Incidence (2020-2023)",
             fontsize=12.5, fontweight="bold", y=1.03)
fig.text(0.02, -0.06,
    "Note. Mann-Kendall test applied independently to each year's incidence series. INC=increasing; "
    "DEC=decreasing; NO=no significant trend. Sen's slope (cases/week) annotated within bars where |\u03c4|>0.12.",
    fontsize=7.5, style="italic", ha="left")
plt.tight_layout()
plt.savefig(f"{OUTPATH}/figure1_mannkendall.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.close()

# FIGURE 2 -- Keyword performance comparison

print("Generating Figure 2...")

fig, ax = plt.subplots(figsize=(12, 7.2))
width = 0.25
kws = ["Measles", "Tigdas", "Tipdas"]
xpos = np.arange(len(YEARS))
for j, kw in enumerate(kws):
    sub = best[best.Keyword == kw].set_index("Year").reindex(YEARS)
    offs = (j - 1) * width
    ax.bar(xpos + offs, sub["rho"], width=width * 0.92, color=KW_COL[kw],
           edgecolor="black", linewidth=0.8, alpha=0.9, label=KW_LABEL[kw])
    for i, (yr, r) in enumerate(sub.iterrows()):
        if pd.isna(r["rho"]):
            continue
        lag_lbl = f"L{int(r['Lag']):+d}"
        if abs(r["rho"]) > 0.15:
            ax.text(xpos[i] + offs, r["rho"] / 2, lag_lbl, ha="center", va="center",
                    fontsize=7.5, color="white", fontweight="bold")
        s = sig_stars(r["p"])
        if s:
            off2 = 0.035 if r["rho"] >= 0 else -0.06
            va = "bottom" if r["rho"] >= 0 else "top"
            ax.text(xpos[i] + offs, r["rho"] + off2, s, ha="center", va=va, fontsize=9, fontweight="bold")

ax.axhline(0, color="black", linewidth=1.1)
ax.axhline(0.3, linestyle="--", color="#555555", linewidth=1, alpha=0.7)
ax.axhline(-0.3, linestyle="--", color="#555555", linewidth=1, alpha=0.7)
ax.text(3.65, 0.33, "\u03c1 = +0.30", fontsize=8.5, color="#555555")
ax.text(3.65, -0.37, "\u03c1 = \u22120.30", fontsize=8.5, color="#555555")

# callout boxes -- placed clear of bars; legend is BELOW the panel so nothing overlaps
ax.annotate("2022: all three keywords\nsimultaneously significant",
            xy=(2.15, 0.45), xytext=(1.55, 0.72),
            fontsize=9, ha="center", family="serif",
            bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#555555", lw=0.8),
            arrowprops=dict(arrowstyle="->", color="#333333", lw=1.1))
ax.annotate("2020: pandemic\ndecoupling",
            xy=(-0.25, -0.74), xytext=(0.55, -0.86),
            fontsize=9, ha="center", family="serif",
            bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#555555", lw=0.8),
            arrowprops=dict(arrowstyle="->", color="#333333", lw=1.1))

ax.set_xticks(xpos); ax.set_xticklabels(YEARS)
ax.set_xlabel("Year"); ax.set_ylabel("Best Spearman \u03c1")
ax.set_ylim(-0.95, 0.85)
ax.set_title("Figure 2. Keyword Performance Comparison: Best Spearman \u03c1 by Search Term and Year (National)",
            fontsize=12, fontweight="bold", pad=14)
ax.grid(axis="y", linestyle="--", color="#cccccc", linewidth=0.5)
ax.set_axisbelow(True)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3, fontsize=9.5, frameon=True)

fig.text(0.02, -0.09,
    "Note. Each bar = maximum |\u03c1| across lags \u22121, 0, +1 week per keyword\u2013year. L\u22121 = search lags cases; "
    "L0 = simultaneous; L+1 = search leads cases (early-warning). * p<0.05, ** p<0.01, *** p<0.001. "
    "Negative 2020 values reflect pandemic-era behavioral decoupling. EN=English; FIL=Filipino.",
    fontsize=7.5, style="italic")
plt.tight_layout()
plt.savefig(f"{OUTPATH}/figure2_keyword_comparison.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.close()

# FIGURE 3 -- Spearman heatmap (4 panels)

print("Generating Figure 3...")

fig, axes = plt.subplots(1, 4, figsize=(16, 4.6), sharey=False)
lag_labels = ["Lag \u22121\n(lags cases)", "Lag 0\n(simultaneous)", "Lag +1\n(leads cases)"]
kw_order = ["Measles", "Tigdas", "Tipdas"]
panel_labels = ["(a) 2020", "(b) 2021", "(c) 2022", "(d) 2023"]
im = None
for k, yr in enumerate(YEARS):
    ax = axes[k]
    sub = sp_df[sp_df.Year == yr]
    mat = np.full((3, 3), np.nan)
    pmat = np.full((3, 3), np.nan)
    for i, kw in enumerate(kw_order):
        for j, lag in enumerate([-1, 0, 1]):
            row = sub[(sub.Keyword == kw) & (sub.Lag == lag)]
            if len(row):
                mat[i, j] = row["rho"].values[0]
                pmat[i, j] = row["p"].values[0]
    im = ax.imshow(mat, cmap="RdYlGn", vmin=-0.85, vmax=0.85, aspect="auto")
    for i in range(3):
        for j in range(3):
            v, p = mat[i, j], pmat[i, j]
            if np.isnan(v):
                continue
            txt = f"{v:+.3f}{' (sig)' if p < 0.05 else ''}"
            color = "white" if abs(v) > 0.5 else "black"
            ax.text(j, i, txt, ha="center", va="center", fontsize=8.5, fontweight="bold", color=color)
            if p < 0.05:
                ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, edgecolor="black", linewidth=2.2))
    ax.set_xticks([0, 1, 2]); ax.set_xticklabels(lag_labels, fontsize=8)
    ax.set_ylim(2.5, -0.5)
    if k == 0:
        ax.set_yticks([0, 1, 2]); ax.set_yticklabels([KW_LABEL[k2] for k2 in kw_order], fontsize=9, fontweight="bold")
    else:
        ax.set_yticks([])
    ax.set_title(panel_labels[k], fontsize=11, fontweight="bold")
    for spine in ax.spines.values():
        spine.set_visible(True); spine.set_color("black"); spine.set_linewidth(1.0)
cbar = fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02)
cbar.set_label("Spearman \u03c1")
fig.suptitle("Figure 3. Spearman Cross-Correlation Coefficients by Keyword, Lag, and Year (National)",
            fontsize=12.5, fontweight="bold", y=1.05)
fig.text(0.02, -0.08,
    "Note. Each cell = Spearman \u03c1 between keyword search volume and national weekly incidence. "
    "Lag \u22121: cases precede search (reactive). Lag 0: simultaneous. Lag +1: search leads cases (early-warning). "
    "Bold border = p<0.05. Negative 2020 values reflect pandemic-era decoupling.",
    fontsize=7.5, style="italic")
plt.savefig(f"{OUTPATH}/figure3_spearman_heatmap.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.close()

# FIGURE 5 -- Combined 4-year incidence + MK trend lines

print("Generating Figure 5...")

nat_sorted = nat.sort_values(["Year", "Week"]).reset_index(drop=True)
nat_sorted["week_idx"] = np.arange(1, len(nat_sorted) + 1)

fig, ax = plt.subplots(figsize=(14, 6))
for yr in YEARS:
    sub = nat_sorted[nat_sorted.Year == yr]
    xi = sub["week_idx"].values
    ax.fill_between(xi, sub["inc"], color=YR_COL[yr], alpha=0.15)
    ax.plot(xi, sub["inc"], color=YR_COL[yr], linewidth=2, label=str(yr))

    r = mk_df[mk_df.Year == yr].iloc[0]
    n = len(sub)
    rown = np.arange(1, n + 1)
    intercept = sub["inc"].mean() - r["slope"] * (n / 2)
    trend_line = intercept + r["slope"] * rown
    lty = "-" if r["p_value"] < 0.05 else "--"
    tcol = "#1a9641" if r["tau"] > 0 else ("#d7191c" if r["tau"] < 0 else "#888888")
    ax.plot(xi, trend_line, color=tcol, linewidth=1.6, linestyle=lty)

    xmid = xi.mean()
    ax.annotate(f"\u03c4={r['tau']:+.2f}{' (sig)' if r['p_value'] < 0.05 else ''}",
                (xmid, sub['inc'].mean() + sub['inc'].std() * 0.8), fontsize=8.5, fontweight="bold",
                ha="center", color=tcol, bbox=dict(boxstyle="round,pad=0.25", fc="white", ec=tcol, lw=0.8))
    if yr < 2023:
        ax.axvline(xi.max() + 0.5, linestyle=":", color="#888888", linewidth=1)

ax.set_xlabel("Continuous Epidemiological Week Index (2020 \u2192 2023)")
ax.set_ylabel("Weekly Measles Incidence (New Cases per Week)")
ax.set_title("Figure 5. National Weekly Measles Incidence with Mann-Kendall Trend Lines (2020-2023)",
             fontsize=12.5, fontweight="bold")
ax.legend(title="Year", loc="upper right", fontsize=9)
ax.grid(axis="y", linestyle="--", color="#dddddd", linewidth=0.5)
fig.text(0.02, -0.05,
    "Note. Shaded areas/lines = weekly incidence. Overlaid lines = Sen's slope trend (solid=p<0.05, dashed=n.s.). "
    "Vertical dotted lines mark year boundaries. The 2020 series begins ~Week 20 due to COVID-19 disruption of "
    "PIDSR surveillance reporting.", fontsize=7.5, style="italic")
plt.tight_layout()
plt.savefig(f"{OUTPATH}/figure5_combined_trend.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.close()

print(f"\nAll figures saved to {os.path.abspath(OUTPATH)}")